# Week 2 Day 1 Advanced Exercise: 3-Way Conversation with Ollama

This notebook completes the Week 2 Day 1 advanced exercise using local Ollama models only. It creates a live, streaming conversation between three AI personas and passes the full transcript into each turn, which is the pattern recommended in the course notebook for 3-way conversations.

Before running the notebook, make sure Ollama is running locally:

```bash
ollama serve
ollama pull llama3.2
```

Optional: pull extra models and change the model names below, for example `mistral`, `qwen2.5`, or `phi4-mini`.

In [ ]:
from IPython.display import Markdown, display
import time
from openai import OpenAI
import requests

OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_HEALTH_URL = "http://localhost:11434/"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

## Check Ollama

This quick check gives a friendly error if Ollama is not running.

In [ ]:
def check_ollama():
    try:
        response = requests.get(OLLAMA_HEALTH_URL, timeout=3)
        response.raise_for_status()
        return True
    except requests.RequestException as exc:
        raise RuntimeError(
            "Ollama does not seem to be running. Start it with `ollama serve`, "
            "then run `ollama pull llama3.2`."
        ) from exc

check_ollama()
display(Markdown("Ollama is running."))

## Personas

The three speakers can use the same local model or different local models. If a model is not available, pull it with `ollama pull <model-name>` or replace it with one you already have.

In [ ]:
participants = [
    {
        "name": "Alex",
        "model": "llama3.2",
        "system": """
You are Alex, a practical product engineer. You care about usefulness, trade-offs, and shipping working prototypes.
You are in a conversation with Blake and Casey. Keep responses concise, specific, and conversational.
Do not speak for Blake or Casey.
""".strip(),
    },
    {
        "name": "Blake",
        "model": "mistral",
        "system": """
You are Blake, a skeptical reviewer. You challenge assumptions, ask for evidence, and point out risks.
You are in a conversation with Alex and Casey. Keep responses concise, sharp, and constructive.
Do not speak for Alex or Casey.
""".strip(),
    },
    {
        "name": "Casey",
        "model": "phi4-mini",
        "system": """
You are Casey, a creative strategist. You connect ideas, suggest unexpected angles, and keep the discussion moving.
You are in a conversation with Alex and Blake. Keep responses concise, imaginative, and grounded.
Do not speak for Alex or Blake.
""".strip(),
    },
]

topic = "How should a small business use a local LLM assistant without overcomplicating the first prototype?"

## Streaming Conversation Helpers

The important part is `build_prompt`: every turn includes the full conversation so far, then asks exactly one speaker to continue. `stream_next_message` then updates a notebook display as tokens arrive from Ollama.

In [ ]:
def render_conversation(turns):
    if not turns:
        return "No one has spoken yet."
    return "\n\n".join(f"{turn['speaker']}: {turn['message']}" for turn in turns)


def build_prompt(speaker, turns):
    names = ", ".join(person["name"] for person in participants)
    transcript = render_conversation(turns)
    return f"""
The conversation topic is:
{topic}

The speakers are: {names}.

The conversation so far is:
{transcript}

Now respond as {speaker['name']} with the next contribution.
Rules:
- Write only {speaker['name']}'s message.
- Do not include the speaker name as a prefix.
- Do not write dialogue for anyone else.
- Keep it to 2-4 sentences.
""".strip()


def stream_next_message(speaker, turns, temperature=0.7):
    messages = [
        {"role": "system", "content": speaker["system"]},
        {"role": "user", "content": build_prompt(speaker, turns)},
    ]
    stream = ollama.chat.completions.create(
        model=speaker["model"],
        messages=messages,
        temperature=temperature,
        stream=True,
    )

    message = ""
    display_handle = display(Markdown(f"### {speaker['name']}\n\n"), display_id=True)

    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        if not delta:
            continue
        message += delta
        display_handle.update(Markdown(f"### {speaker['name']}\n\n{message}"))

    final_message = message.strip()
    display_handle.update(Markdown(f"### {speaker['name']}\n\n{final_message}"))
    return final_message


def show_turn(speaker, message):
    display(Markdown(f"### {speaker}\n\n{message}"))

## Run the Live 3-Way Conversation

This runs four rounds. Each round gives Alex, Blake, and Casey one turn, and each reply streams into the notebook as it is generated.

In [ ]:
turns = [
    {
        "speaker": "Moderator",
        "message": f"Let's discuss this question: {topic}",
    }
]

display(Markdown(f"## Topic\n\n{topic}"))
show_turn(turns[0]["speaker"], turns[0]["message"])

rounds = 4

for _ in range(rounds):
    for speaker in participants:
        message = stream_next_message(speaker, turns)
        turns.append({"speaker": speaker["name"], "message": message})
        time.sleep(0.25)

## Full Transcript

This final cell is useful if you want the whole conversation as one clean block.

In [ ]:
display(Markdown("## Full transcript\n\n" + render_conversation(turns).replace("\n", "  \n")))

## Variations to Try

- Change the `topic` to a debate, product idea, or business question.
- Use three different Ollama models by changing each participant's `model` value.
- Change the persona prompts so one participant is optimistic, one skeptical, and one technical.
- Increase or decrease `rounds` to control the length of the conversation.